# GEMM cache-locality & efficiency visualizer

Reads the tidy CSVs produced by `scripts/parse_likwid.py` from `output/<processor>/`
and saves one PNG per figure into the same per-processor folder, plus cross-processor
comparisons into `output/comparison/`.

Run order:
1. `bash scripts/run_bench.sh` on each node (or `scripts/submit_all.sh`)
2. `python3 scripts/parse_likwid.py`
3. this notebook, top to bottom

In [8]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import matplotlib.pyplot as plt

OUT = Path('output') if Path('results').is_dir() else Path('..') / 'output'
OUT = OUT.resolve()

def load(proc_dir):
    d = {}
    p = proc_dir / 'meta.csv'
    if p.exists():
        rows = [l.strip().split(',', 1) for l in p.read_text().splitlines() if ',' in l]
        d['meta'] = dict(rows)
    for name in ('summary', 'gemm', 'likwid', 'likwid_perthread'):
        p = proc_dir / (name + '.csv')
        if p.exists():
            d[name] = pd.read_csv(p)
    return d

procs = {}
for d in sorted(OUT.iterdir()):
    if d.is_dir() and d.name != 'comparison' and any((d/f).exists() for f in
            ('summary.csv','gemm.csv','likwid.csv')):
        procs[d.name] = load(d)
print('processors found:', list(procs))

processors found: ['GNR', 'ICX', 'SPR', 'nvidia_grace_H100']


## Setup — locate `output/` and load the tidy CSVs per processor

## Helper: cache-hierarchy marker lines

The x-axis is the cube size N (M=N=K); a vertical line is drawn where the
working set  (A+B+C = 3·N²·element_size)  crosses each cache level.

In [9]:
def cache_crossings(meta, elem_bytes):
    xs = {}
    for key, label in (('l1d_bytes','L1'), ('l2_bytes','L2'), ('l3_bytes','L3')):
        b = meta.get(key) if isinstance(meta, dict) else None
        try: b = float(b)
        except (TypeError, ValueError): continue
        if b > 0:
            xs[label] = int((b / (3 * elem_bytes)) ** 0.5)
    return xs

def add_cache_bands(ax, xs):
    band_colors = {'L1': 'tab:green', 'L2': 'gold', 'L3': 'tab:orange'}
    prev, first = 0, True
    for label, x in sorted(xs.items(), key=lambda kv: kv[1]):
        ax.axvspan(prev, x, alpha=0.07,
                   color=band_colors.get(label, 'tab:gray'))
        ax.axvline(x, ls=':', c='gray', lw=1)
        ax.text(x, ax.get_ylim()[1], ' '+label, va='top', ha='left', fontsize=8, color='gray')
        prev, first = x, False

## Per-processor plots

Saved into `output/<processor>/*.png`.

In [10]:
LEVEL_ORDER = ['V0','V1','V2','V3','V4','V5','mkl','aocl','openblas','BLAS']
def level_style(name):
    for i, k in enumerate(LEVEL_ORDER):
        if name.lower().startswith(k.lower()):
            return dict(color=plt.cm.viridis(i/7.0), marker='o', lw=2)
    return dict(color='tab:purple', marker='s', lw=1.5)

for proc, data in procs.items():
    g = data.get('gemm')
    if g is None or not len(g):
        print(f'{proc}: no gemm data'); continue
    meta = data.get('meta', {})
    for dt, sub in g.groupby('dtype'):
        esz = {'FP32':4,'FP64':8,'FP16':2,'BF16':2}[dt]
        unit = sub['unit'].iloc[0] if 'unit' in sub else 'GFlop/s'
        fig, ax = plt.subplots(figsize=(8,5))
        for lvl, lsub in sub.groupby('level'):
            lsub = lsub.sort_values('cube')
            if lsub['cube'].nunique() < 2:      # single size -> markers
                ax.scatter(lsub['cube'], lsub['perf'], **level_style(lvl),
                           label=lvl, s=60, zorder=3)
            else:
                ax.plot(lsub['cube'], lsub['perf'], **level_style(lvl), label=lvl)
        xs = cache_crossings(meta, esz)
        if xs: add_cache_bands(ax, xs)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('matrix size N (cube M=N=K)'); ax.set_ylabel(f'{unit}  (log)')
        ax.set_title(f'{proc} — cache locality ladder, {dt}')
        ax.legend(fontsize=8, ncol=2); fig.tight_layout()
        fig.savefig(OUT/proc/f'gemm_ladder_{dt}.png'); plt.close(fig)
    print(f'{proc}: gemm_ladder_*.png')

GNR: gemm_ladder_*.png
ICX: gemm_ladder_*.png
SPR: gemm_ladder_*.png
nvidia_grace_H100: gemm_ladder_*.png


## LIKWID counters vs matrix size (SP/DP rates, energy, memory traffic)

Only present for runs with `USE_LIKWID=1`.

In [11]:
FLOP_METRICS = ['SP [MFLOP/s]', 'DP [MFLOP/s]','AVX512 SP', 'AVX512 DP', 'BF16 [MFLOP/s]', 'FP rate [MFLOP/s]']

for proc, data in procs.items():
    L = data.get('likwid')
    if L is None or not len(L):
        print(f'{proc}: no likwid data'); continue
    meta = data.get('meta', {})
    def col(*names):
        for n in names:
            if n in L.columns: return n
        return None
    sp_c, dp_c = col('AVX512 SP [MFLOP/s]'), col('AVX512 DP [MFLOP/s]')
    bf_c = col('BF16 [MFLOP/s]')
    fp_c = col('FP rate [MFLOP/s]')
    e_pkg, e_dram = col('Energy PKG [J]'), col('Energy DRAM [J]')
    e_tot = col('Energy [J]') or e_pkg
    bw, vol = col('Memory bandwidth [MBytes/s]'), col('Memory data volume [GBytes]')
    rt = col('Runtime (RDTSC) [s]')
    dt_c = 'dtype' if 'dtype' in L.columns else None

    # ---- flops + energy + memory in one profile figure, by dtype ----
    for dt, sub in L.groupby(dt_c):
        sub = sub.sort_values('size')
        fig, axs = plt.subplots(1, 3, figsize=(14, 4))
        for name, c in (('SP', sp_c), ('DP', dp_c), ('BF16', bf_c)):
            if c and sub[c].notna().any():
                axs[0].plot(sub['size'], sub[c]/1e3, marker='o', label=f'{name} MFLOP/s→GFLOP/s')
        if fp_c and sub[fp_c].notna().any():
            axs[0].plot(sub['size'], sub[fp_c]/1e3, marker='o', label='FP (combined)')
        for lbl, c in (('PKG', e_pkg), ('DRAM', e_dram), ('total', e_tot)):
            if c and sub[c].notna().any():
                axs[1].plot(sub['size'], sub[c], marker='s', label=f'energy {lbl}')
        for lbl, c in (('read', col('Memory read bandwidth [MBytes/s]')),
                       ('write', col('Memory write bandwidth [MBytes/s]')),
                       ('total', bw)):
            if c and sub[c].notna().any():
                axs[2].plot(sub['size'], sub[c], marker='^', label=f'mem {lbl}')
        if vol and vol in sub:
            axs[2].plot(sub['size'], sub[vol], marker='v', label='volume GB')
        for ax in axs:
            ax.set_xscale('log'); ax.set_xlabel('matrix size N'); ax.grid(alpha=.3)
            if ax.get_legend_handles_labels()[0]: ax.legend(fontsize=7)
        axs[0].set_ylabel('GFLOP/s'); axs[1].set_ylabel('J')
        axs[2].set_ylabel('MBytes/s / GBytes')
        fig.suptitle(f'{proc} — LIKWID profile, {dt}')
        fig.tight_layout()
        fig.savefig(OUT/proc/f'likwid_profile_{dt}.png'); plt.close(fig)
        plt.show()
    print(f'{proc}: likwid_profile_*.png')

GNR: likwid_profile_*.png
ICX: likwid_profile_*.png
SPR: likwid_profile_*.png
nvidia_grace_H100: no likwid data


## Thread-level load balance (violin of per-thread FLOP rate)

In [13]:
import re
import matplotlib.pyplot as plt
from pathlib import Path

for proc, data in procs.items():
    PT = data.get('likwid_perthread')
    if PT is None or PT.empty:
        continue

    sel = PT[PT['metric'].isin(FLOP_METRICS)]
    if sel.empty:
        continue

    proc_dir = Path(OUT) / str(proc)
    proc_dir.mkdir(parents=True, exist_ok=True)

    for (dt, size, metric), sub in sel.groupby(['dtype', 'size', 'metric']):
        vals = sub['value'].dropna().to_numpy()
        if len(vals) == 0:
            continue

        fig, ax = plt.subplots(figsize=(6, 4))

        # Handle zero-variance / single-sample cases where KDE fails
        if len(vals) < 2 or vals.min() == vals.max():
            ax.boxplot([vals], showmeans=True)
        else:
            ax.violinplot([vals], showmeans=True)

        # Sanitize metric string for filesystem safety (replace slashes and spaces)
        safe_metric = re.sub(r'[^a-zA-Z0-9_\-]+', '_', str(metric)).strip('_')

        ax.set_title(f'{proc} {dt} N={size} — {metric} per thread')
        ax.set_ylabel('MFLOP/s')
        ax.set_xticks([])
        fig.tight_layout()

        out_path = proc_dir / f'threads_{dt}_{size}_{safe_metric}.png'
        fig.savefig(out_path)
        plt.close(fig)

    print(f'{proc}: generated thread plots in {proc_dir}')

GNR: generated thread plots in /home/hpc/ihpc/ihpc171h/benchmark-collection/output/GNR
ICX: generated thread plots in /home/hpc/ihpc/ihpc171h/benchmark-collection/output/ICX
SPR: generated thread plots in /home/hpc/ihpc/ihpc171h/benchmark-collection/output/SPR


## Cross-processor comparison

In [14]:
comp = OUT / 'comparison'; comp.mkdir(exist_ok=True)
frames = []
for proc, data in procs.items():
    S = data.get('summary')
    if S is not None and len(S): frames.append(S)
if frames:
    allS = pd.concat(frames)
    g = allS.groupby(['proc','dtype'])['gemm_best_gflops'].max().unstack()
    g.plot(kind='bar', rot=20, title='best GEMM GFlop/s per processor')
    plt.ylabel('GFlop/s'); plt.tight_layout()
    plt.savefig(comp/'compare_gemm_peak.png'); plt.close()

    if 'Energy_PKG_J' in allS or 'Energy_J' in allS:
        e = 'Energy_PKG_J' if 'Energy_PKG_J' in allS else 'Energy_J'
        tmp = allS.dropna(subset=[e, 'gemm_best_gflops'])
        tmp = tmp[tmp[e] > 0]
        tmp['gflops_per_joule'] = tmp['gemm_best_gflops'] / tmp[e]
        m = tmp.groupby(['proc','dtype'])['gflops_per_joule'].mean().unstack()
        m.plot(kind='bar', rot=20, title='energy efficiency (GFlop/s per Joule, mean over sizes)')
        plt.tight_layout(); plt.savefig(comp/'compare_efficiency.png'); plt.close()

    bcol = 'Memory_bandwidth_MBytes/s'
    if bcol in allS and allS[bcol].notna().any():
        mb = allS.dropna(subset=[bcol]).groupby(['proc','dtype'])[bcol].max().unstack()
        mb.plot(kind='bar', rot=20, title='socket memory bandwidth (MBytes/s, max over sweep)')
        plt.tight_layout(); plt.savefig(comp/'compare_memory.png'); plt.close()
    allS.to_csv(comp/'all_summary.csv', index=False)
    print('comparison ->', comp)

comparison -> /home/hpc/ihpc/ihpc171h/benchmark-collection/output/comparison
